[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week0_setup_check_STARTER.ipynb)


# Week 0 -- Before You See (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** environment setup, image formats (JPEG/PNG/DICOM), tensor shapes, dataset verification

**Dataset:** Setup only

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


In [ ]:
# -- Install the Full CV Stack -- run once, then restart kernel -------------
# torch        -- PyTorch: tensors, autograd, neural network training
# torchvision  -- image transforms, pretrained models (ResNet, etc.)
# timm         -- 600+ pretrained CV architectures (EfficientNet, ViT)
# monai        -- Medical Open Network for AI: DICOM, stain normalisation
# opencv-python -- low-level image ops: frame extraction, color spaces
# Pillow       -- PIL Image: standard image file loading (JPEG, PNG)
# captum       -- PyTorch interpretability (Integrated Gradients, SHAP)
# lime         -- Local Interpretable Model-agnostic Explanations
# shap         -- SHapley Additive exPlanations
# grad-cam     -- GradCAM and 10+ saliency methods for CNNs
# torchstain   -- stain normalisation for histopathology (Macenko, Vahadane)

!pip install torch torchvision timm monai opencv-python Pillow \
             matplotlib seaborn scikit-learn tqdm \
             captum lime shap grad-cam torchstain -q
print('Installation complete. Restart the kernel before importing.')


In [ ]:
# -- Day 2: Three Image Formats -- What Each Returns as a Python Object -----
from PIL import Image          # PIL: loads JPEG, PNG, TIFF
import numpy as np             # NumPy: inspect pixel arrays
import pydicom                 # pydicom: reads DICOM medical files
import torchvision.transforms as T  # torchvision: PIL -> tensor

# == JPEG: natural photograph ================================================
# Image.open() returns a PIL Image object, not a NumPy array
# .convert('RGB') ensures 3 channels even if the file is grayscale
try:
    jpeg_pil = Image.open('datasets/pets/sample.jpg').convert('RGB')
except FileNotFoundError:
    # Synthetic fallback if dataset not yet downloaded
    jpeg_pil = Image.fromarray((np.random.rand(224,224,3)*255).astype('uint8'))
    print('NOTE: using synthetic image -- replace with real dataset')

# np.array() converts PIL Image to NumPy HWC array (Height x Width x Channels)
# PIL reports size as (Width, Height) -- NumPy reverses this to (H, W, C)
jpeg_arr = np.array(jpeg_pil)
print('=== JPEG (Oxford Pets sample) ===')
print(f'  PIL .size (W, H):       {jpeg_pil.size}')  # PIL: (W,H)
print(f'  NumPy .shape (H, W, C): {jpeg_arr.shape}') # NumPy: (H,W,C)
print(f'  dtype:                  {jpeg_arr.dtype}')  # uint8 = values 0-255
print(f'  value range:            [{jpeg_arr.min()}, {jpeg_arr.max()}]')
print()

# == DICOM: medical chest X-ray =============================================
# pydicom.dcmread() reads the FULL DICOM record:
#   - Patient metadata (PatientID, DOB -- de-identified in research datasets)
#   - Acquisition parameters (Modality, WindowCenter, BitsAllocated)
#   - The pixel array (the actual image)
try:
    ds = pydicom.dcmread('datasets/xray/sample.dcm')
    xray_arr = ds.pixel_array   # extract ONLY the pixel values
except FileNotFoundError:
    xray_arr = (np.random.rand(512,512)*65535).astype('uint16')
    print('NOTE: using synthetic DICOM -- replace with real dataset')

print('=== DICOM (Chest X-ray) ===')
print(f'  Shape:       {xray_arr.shape}')   # (H,W) -- single channel
print(f'  dtype:       {xray_arr.dtype}')   # uint16 -- NOT uint8!
print(f'  value range: [{xray_arr.min()}, {xray_arr.max()}]')
print()

# == KEY INSIGHT ============================================================
# JPEG: uint8, range [0,255], 3 channels (RGB), lossy compression
# DICOM: uint16, range [0,65535], 1 channel (grayscale), medical standard
# NEVER divide DICOM by 255 -- that destroys the 12-bit diagnostic range
# Correct DICOM normalisation: (arr - arr.min()) / (arr.max() - arr.min())
print('KEY LESSON:')
print('  JPEG  -> divide by 255.0 to normalise to [0.0, 1.0]')
print('  DICOM -> use windowing or (arr-min)/(max-min) -- NEVER divide by 255')


In [ ]:
# -- Day 3: CHW Format -- Why PyTorch Is Channels-First ---------------------
# PIL/NumPy return HWC: (Height, Width, Channels) e.g. (224, 224, 3)
# PyTorch expects CHW: (Channels, Height, Width) e.g. (3, 224, 224)
# Mixing these up is the most common silent shape error in CV.

from PIL import Image
import torchvision.transforms as T
import numpy as np, torch

try:
    img = Image.open('datasets/pets/sample.jpg').convert('RGB')
except FileNotFoundError:
    img = Image.fromarray((np.random.rand(300,400,3)*255).astype('uint8'))

# -- PIL -> NumPy: gives HWC ------------------------------------------------
img_hwc = np.array(img)
print(f'PIL -> NumPy (HWC): {img_hwc.shape}')   # e.g. (300, 400, 3)
print(f'  H={img_hwc.shape[0]}, W={img_hwc.shape[1]}, C={img_hwc.shape[2]}')
print(f'  dtype: {img_hwc.dtype}  range: [{img_hwc.min()}, {img_hwc.max()}]')
print()

# -- PIL -> T.ToTensor(): gives CHW float32 in [0,1] -----------------------
# T.ToTensor() does THREE things at once:
#   1. Reorders axes:     HWC -> CHW
#   2. Changes dtype:     uint8 -> float32
#   3. Rescales values:   [0, 255] -> [0.0, 1.0]
img_chw = T.ToTensor()(img)
print(f'PIL -> T.ToTensor() (CHW): {img_chw.shape}')  # e.g. (3, 300, 400)
print(f'  C={img_chw.shape[0]}, H={img_chw.shape[1]}, W={img_chw.shape[2]}')
print(f'  dtype: {img_chw.dtype}  range: [{img_chw.min():.3f}, {img_chw.max():.3f}]')
print()

# -- Adding the batch dimension --------------------------------------------
# Models expect (Batch, C, H, W) = BCHW format
# .unsqueeze(0) inserts a new dimension of size 1 at position 0
img_bchw = img_chw.unsqueeze(0)   # (C,H,W) -> (1,C,H,W)
print(f'With batch dim (BCHW): {img_bchw.shape}')   # e.g. (1,3,300,400)
print(f'  B=1 (single image), C={img_bchw.shape[1]}, H={img_bchw.shape[2]}, W={img_bchw.shape[3]}')
print()

# -- STOP AND TRACE --------------------------------------------------------
print('=== STOP AND TRACE ===')
print('Predict the output shape of: nn.Conv2d(3, 32, kernel_size=3, padding=1)')
print('Input: (batch=2, C=3, H=224, W=224)')
print('Formula: H_out = (H_in + 2*padding - kernel_size) / stride + 1')
h_out = (224 + 2*1 - 3) // 1 + 1
print(f'H_out = (224 + 2*1 - 3) / 1 + 1 = {h_out}')
print(f'Full output shape: (2, 32, {h_out}, {h_out})')
print('padding=1 with kernel=3 preserves spatial dimensions -- this is intentional.')


In [ ]:
# -- Friday Build: Environment Verification --------------------------------
print('=' * 55)
print('BOOK 3 -- The Computer Vision Internship')
print('Week 0: Environment Verification')
print('=' * 55)

all_passed = True

# Test 1: PyTorch version --------------------------------------------------
# We need PyTorch 2.0+ for all features used in this course
try:
    import torch
    major = int(torch.__version__.split('.')[0])
    assert major >= 2, f'PyTorch 2.0+ required, got {torch.__version__}'
    print(f'  PASS  Test 1 -- PyTorch {torch.__version__}')
except AssertionError as e:
    print(f'  FAIL  Test 1 -- {e}'); all_passed = False

# Test 2: GPU availability ------------------------------------------------
# GPU is optional for Weeks 0-4, strongly recommended from Week 5
import torch
cuda = torch.cuda.is_available()
level = 'PASS' if cuda else 'WARN'
print(f'  {level}  Test 2 -- GPU available: {cuda}')
if cuda:
    print(f'           GPU: {torch.cuda.get_device_name(0)}')
else:
    print('           No GPU -- Weeks 5-12 will be slow. Colab: Runtime -> Change runtime type -> T4')

# Test 3: MONAI (medical imaging) -----------------------------------------
# Required for DICOM loading, stain normalisation (Weeks 6-8)
try:
    import monai
    print(f'  PASS  Test 3 -- MONAI {monai.__version__}')
except ImportError:
    print('  FAIL  Test 3 -- MONAI not installed'); all_passed = False

# Test 4: XAI libraries (required for Week 10) ----------------------------
for lib in ['captum', 'shap', 'lime']:
    try:
        __import__(lib)
        print(f'  PASS  Test 4 -- {lib}')
    except ImportError:
        print(f'  FAIL  Test 4 -- {lib} not installed'); all_passed = False

# Test 5: transform pipeline works ----------------------------------------
# Catch mismatched torchvision/PyTorch versions early
try:
    import torchvision.transforms as T
    from PIL import Image; import numpy as np
    tf  = T.Compose([T.Resize(224), T.ToTensor()])
    img = Image.fromarray(np.zeros((100,100,3), dtype='uint8'))
    t   = tf(img)
    assert t.shape == (3, 224, 224), f'Wrong shape: {t.shape}'
    print('  PASS  Test 5 -- torchvision transforms')
except Exception as e:
    print(f'  FAIL  Test 5 -- {e}'); all_passed = False

print()
print('=' * 55)
print('All tests PASSED -- ready for Week 1!' if all_passed else 'Some tests FAILED -- fix issues above.')
print('=' * 55)


## Learning Objectives

By the end of this week, you will be able to:

- Install and verify the full CV stack: PyTorch, torchvision, timm, monai, opencv-python
- Load a JPEG, PNG, and DICOM file and explain what each returns as a Python object
- Convert an image to a tensor in CHW format and explain why PyTorch uses CHW not HWC
- Run three environment tests that confirm your setup is ready for Week 1
- Build the project folder structure and save your first requirements.txt



---

## MONDAY -- "Installing the CV Stack"


CV requires a different library ecosystem from NLP or tabular ML. You install the core stack today and understand what each library does before you use it.


### Exercise 0.1 -- Environment install

Run the install command above. Verify: python -c "import torch, torchvision, timm, monai, cv2; print('Stack ready')"


In [ ]:
# Exercise 0.1: Environment install
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


### Exercise 0.2 -- Format comparison

Load the same image in PIL, OpenCV, and torchvision. Print the shape, dtype, and value range for each. Why are they different?


In [ ]:
# Exercise 0.2: Format comparison
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: Installing the CV Stack (scaffold) --
pip install torch torchvision timm monai opencv-python Pillow matplotlib seaborn scikit-learn tqdm captum lime shap grad-cam


### Monday Morning Moment

*MediVision AI — Monday, 9:30am.*

**Dr. Kwame Asante:** Can you load a DICOM file?

**Ngozi Eze-Okafor:** I can load it. I'm not sure I understand what pixel_array.dtype=uint16 means yet.

**Dr. Kwame Asante:** Good. That's the right uncertainty. A chest X-ray is stored in 12-bit or 16-bit depth — not 8-bit like a JPEG. If you normalise it the same way you normalise a cat photo, you will scale away the diagnostic signal. Write that in a markdown cell.

**Ngozi Eze-Okafor:** Is that one of the common mistakes?

**Dr. Kwame Asante:** It is the most common mistake. It takes most people until Week 6 to discover it. You just found it in Week 0.




---

## TUESDAY -- "Three Image Formats"


JPEG, PNG, and DICOM are not interchangeable. JPEG uses lossy compression — pixel values change on save. PNG is lossless. DICOM is a medical standard that wraps pixel data inside a structured record containing patient metadata, acquisition parameters, and modality information.


### Exercise 0.3 -- DICOM inspection

Load sample.dcm using pydicom. Print: PatientID, Modality, pixel_array.shape, pixel_array.dtype, pixel_array.min(), pixel_array.max(). What does uint16 mean for normalisation?


In [ ]:
# Exercise 0.3: DICOM inspection
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: Three Image Formats (scaffold) --
import pydicom
ds = pydicom.dcmread("sample.dcm")
print(ds.PatientID, ds.Modality)
array = ds.pixel_array  # numpy array, not uint8
print(array.dtype, array.shape)



---

## WEDNESDAY -- "Tensor Shapes and CHW Format"


PyTorch expects images as tensors of shape (C, H, W) — channels first. PIL and OpenCV return (H, W, C) — channels last. This mismatch causes the first silent error most CV beginners encounter. A tensor that looks right but has shape (224, 224, 3) instead of (3, 224, 224) will produce wrong results without an error message.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: Tensor Shapes and CHW Format (scaffold) --
from PIL import Image
import torchvision.transforms as T
img = Image.open("cat.jpg")          # HWC, uint8
tensor = T.ToTensor()(img)           # CHW, float32, [0,1]
print(f"PIL shape: {img.size}")      # (W, H) — note: reversed!
print(f"Tensor shape: {tensor.shape}")  # (3, H, W)



---

## THURSDAY -- "Project Folder Structure"


Structure your project folder now. Every path in every notebook references this structure. Build it once, correctly.


### Exercise 0.4 -- CHW conversion

Load a JPEG image. Convert to a PyTorch tensor. Print its shape before and after. Why does PIL return (W, H) but tensors are (C, H, W)?


In [ ]:
# Exercise 0.4: CHW conversion
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: Project Folder Structure (scaffold) --
book3-cv/
  datasets/
    pets/         # Oxford-IIIT Pets
    xray/         # NIH Chest X-Ray14
    breakhis/     # Breast cancer histopathology
    tcga/         # Uterine cancer patches
    ucf101/       # Video clips
    ham10000/     # Capstone skin lesion
  models/
  outputs/
  notebooks/
  requirements.txt



---

## FRIDAY -- "The Friday Build: Setup Verification"


Your Week 0 deliverable is a setup verification notebook that anyone on the team can clone and run. It must contain all three passing tests with output cells visible, the shapes of one loaded tensor per format (JPEG, PNG, DICOM), and one observation you made about the data that was not in the instructions.


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: Setup Verification (scaffold) --
# All three tests must produce green output before Week 1
print("Test 1 — PyTorch:", torch.__version__)
print("Test 2 — CUDA:", torch.cuda.is_available())
print("Test 3 — MONAI:", monai.__version__)


### Friday Workplace Moment

*MediVision AI — Friday standup.*

**Dr. Kwame Asante:** What did you notice about the data that was not in the instructions?

**Ngozi Eze-Okafor:** The DICOM file contains the patient's name in plain text in the metadata. The chest X-ray dataset says it's de-identified but the field exists. I didn't read the value — but the field exists.

**Dr. Kwame Asante:** Write that down. That is Week 11's fairness audit — not the accuracy, the privacy. You found the right thing to be uncomfortable about.



## YOUR WEEK 0 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] All three setup tests pass — PyTorch version, CUDA availability, MONAI version.
- [ ] I can load JPEG, PNG, and DICOM files and explain what each returns as a Python object.
- [ ] I know why CHW ≠ HWC and what goes wrong if I confuse them.
- [ ] I know why a uint16 DICOM pixel array cannot be normalised the same way as a uint8 JPEG.
- [ ] My project folder is structured and my requirements.txt is saved.


## Challenge Task

> **Core path:** Load the same chest X-ray as uint8 (cast to np.uint8) and as uint16 (native). Display both. How much diagnostic information is lost in the uint8 version?

> **Advanced path:** Read the DICOM metadata spec for the pixel_array field. What is the difference between BitsAllocated, BitsStored, and HighBit? Why do all three exist?


## Common Mistakes

**Normalising a uint16 DICOM as if it were uint8:** pixel_array / 255 destroys 12-bit X-ray data. Normalise to [0,1] using the actual min/max: (arr - arr.min()) / (arr.max() - arr.min()).

**Using PIL for DICOM files:** PIL cannot read DICOM. Use pydicom.dcmread() then .pixel_array for the numpy array.

**Forgetting to check tensor shape after transforms:** ToTensor() produces (C,H,W). A single-channel grayscale X-ray produces (1,H,W). A model expecting (3,H,W) will fail silently or throw a shape error on the first batch.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
